# Lab 3　神奇糖果（中）：用 EIF 把一階誤差扣掉——AIPW 與 TMLE

Lab 2 的 plug-in 偏了 3.4，而且偏差跟著 Q̂ 走。怎麼辦？先問一個更基本的問題：資料多一個人，答案會動多少？把這個「動多少」寫成公式，就是 efficient influence function（EIF）。有了它，修正方法就自然冒出來了：加回去（AIPW），或者推 Q̂（TMLE）。對應投影片 54–68。

> **怎麼跑：** 這本筆記本用 **R** 執行（Colab 會自動選 R runtime；如果右上角不是 R，到「執行階段 → 變更執行階段類型」選 R）。從上往下依序執行：第一格下載資料、第二格載入工具箱，之後每一格都可以改了再跑。想一次跑完：「執行階段 → 全部執行」。
>
> 網頁版（不用登入、直接在瀏覽器跑）：[https://kiwi710680.github.io/Pokemon-CP/03_eif.html](https://kiwi710680.github.io/Pokemon-CP/03_eif.html)

In [ ]:
# Setup: download the data used by this lab (from https://raw.githubusercontent.com/kiwi710680/Pokemon-CP/main/) and set plot size
BASE <- "https://raw.githubusercontent.com/kiwi710680/Pokemon-CP/main/"
files <- c()
for (f in files) if (!file.exists(f)) {
  dir.create(dirname(f), recursive = TRUE, showWarnings = FALSE)
  for (try in 1:3) {                                   # retry: a single network hiccup should not stop the lab
    ok <- tryCatch(download.file(paste0(BASE, f), f, quiet = TRUE, mode = "wb") == 0, error = function(e) FALSE)
    if (ok) break else Sys.sleep(2)
  }
  if (!ok) stop("could not download ", f, " - check the internet connection and run this cell again")
}
options(repr.plot.width = 12, repr.plot.height = 7, repr.plot.res = 100, width = 110, digits = 4)
cat(if (length(files)) paste("ready:", length(files), "data file(s)") else "ready: this lab simulates its own data", "\n")

## 工具箱

這一頁用到的 R 函式都在下面這一格：估計器、資料讀取、畫圖小工具。先執行它；想知道某個函式做什麼，回來這裡找。

In [ ]:
# ---- estimators: one set of nuisance predictions -> different answers ----
# Notation (same as the slides): O = (X, A, Y)
#   Q(a, X) = E[Y | A = a, X]            outcome regression   -> nu$Q1, nu$Q0
#   g(X)    = P(A = 1 | X)               propensity score     -> nu$g
#   H(A, X) = A / g - (1 - A) / (1 - g)  clever covariate
#   D(O; g, Q, psi) = H (Y - Q_A) + Q1 - Q0 - psi   efficient influence function of the ATE
expit <- function(x) 1 / (1 + exp(-x))
bound <- function(p, lo = 0.025, hi = 0.975) pmin(pmax(p, lo), hi)

est_plugin <- function(nu) mean(nu$Q1 - nu$Q0)               # g-computation: predict twice, subtract, average

est_ipw_ht <- function(dat, nu)                               # Horvitz-Thompson IPTW: divide by n
  mean(dat$A * dat$Y / nu$g - (1 - dat$A) * dat$Y / (1 - nu$g))

est_ipw <- function(dat, nu) {                                # Hajek (stabilized) IPTW: divide by the sum of weights
  w1 <- dat$A / nu$g; w0 <- (1 - dat$A) / (1 - nu$g)
  sum(w1 * dat$Y) / sum(w1) - sum(w0 * dat$Y) / sum(w0)
}

eif <- function(dat, nu, psi) {
  H  <- dat$A / nu$g - (1 - dat$A) / (1 - nu$g)
  QA <- ifelse(dat$A == 1, nu$Q1, nu$Q0)
  H * (dat$Y - QA) + nu$Q1 - nu$Q0 - psi
}

wald <- function(psi, D) {                                    # SE from the EIF: sqrt(mean(D^2) / n)
  se <- sqrt(mean(D^2) / length(D))
  c(est = psi, se = se, lo = psi - 1.96 * se, hi = psi + 1.96 * se)
}

est_aipw <- function(dat, nu) {                               # one-step: psi_plug + P_n D  (solves P_n D = 0)
  psi <- mean(eif(dat, nu, 0))
  wald(psi, eif(dat, nu, psi))
}

est_tmle <- function(dat, nu, ybounds = range(dat$Y)) {       # targeting: move Q-hat along H until P_n D = 0
  lo <- ybounds[1]; hi <- ybounds[2]
  sc <- function(y) pmin(pmax((y - lo) / (hi - lo), 1e-6), 1 - 1e-6)
  Ys <- (dat$Y - lo) / (hi - lo)
  Q1s <- sc(nu$Q1); Q0s <- sc(nu$Q0); QAs <- ifelse(dat$A == 1, Q1s, Q0s)
  H  <- dat$A / nu$g - (1 - dat$A) / (1 - nu$g)
  fl <- suppressWarnings(glm(Ys ~ -1 + H, offset = qlogis(QAs), family = quasibinomial()))
  eps <- unname(coef(fl))
  Q1t <- expit(qlogis(Q1s) + eps / nu$g)
  Q0t <- expit(qlogis(Q0s) - eps / (1 - nu$g))
  nu_t <- data.frame(Q1 = lo + (hi - lo) * Q1t, Q0 = lo + (hi - lo) * Q0t, g = nu$g)
  psi <- mean(nu_t$Q1 - nu_t$Q0)
  c(wald(psi, eif(dat, nu_t, psi)), eps = eps)
}

weight_summary <- function(dat, nu) {                         # largest weight and effective sample size (ESS)
  w <- ifelse(dat$A == 1, 1 / nu$g, 1 / (1 - nu$g))
  c(max_weight = max(w),
    ESS_treated = sum(w[dat$A == 1])^2 / sum(w[dat$A == 1]^2), n_treated = sum(dat$A),
    ESS_control = sum(w[dat$A == 0])^2 / sum(w[dat$A == 0]^2), n_control = sum(1 - dat$A))
}

# ---- magic candy: simulated data, so the truth (Q, g, psi) is known ----
species_levels <- c("Pidgey", "Weedle", "Caterpie", "Eevee")

simulate_candy <- function(n = 800, seed = NULL, strength = 1) {
  if (!is.null(seed)) set.seed(seed)
  species <- factor(sample(species_levels, n, replace = TRUE, prob = c(.40, .20, .15, .25)), levels = species_levels)
  top <- c(Pidgey = 450, Weedle = 260, Caterpie = 260, Eevee = 650)[as.character(species)]
  cp  <- round(10 + top * rbeta(n, 2, 2.5))
  mult <- c(Pidgey = 1.85, Weedle = 1.10, Caterpie = 1.08, Eevee = 2.60)[as.character(species)]
  f0   <- mult * cp * (1 - cp / 2600)                         # evolution curve, bends at high CP
  # trainers prefer strong Pokemon and Eevee: confounding. Larger strength -> g closer to 0 or 1
  g0 <- expit(strength * (-1.8 + 3.2 * (cp / top)) + 0.5 * (species == "Eevee"))
  A  <- rbinom(n, 1, g0)
  tau <- 20 + 0.30 * cp                                       # candy effect: stronger Pokemon gain more
  Y1 <- f0 + tau + rnorm(n, 0, 30)
  Y0 <- f0 + rnorm(n, 0, 30)
  data.frame(species, cp, A, Y = ifelse(A == 1, Y1, Y0), g0, Q1 = f0 + tau, Q0 = f0, Y1, Y0)
}

candy_truth <- function(n = 2e6, seed = 1) {                  # population ATE = E[tau(X)]
  set.seed(seed)
  species <- sample(species_levels, n, replace = TRUE, prob = c(.40, .20, .15, .25))
  top <- c(Pidgey = 450, Weedle = 260, Caterpie = 260, Eevee = 650)[species]
  mean(20 + 0.30 * round(10 + top * rbeta(n, 2, 2.5)))
}
candy_population <- function(n = 20000, seed = 2027, strength = 1) simulate_candy(n, seed, strength)

# ---- nuisance learners ----
knn_Q <- function(train, newdata, a, k = 10) {                # same species, same arm, k nearest CP, average Y
  out <- rep(NA_real_, nrow(newdata))
  for (s in levels(newdata$species)) {
    tr <- train[train$species == s & train$A == a, ]
    id <- which(newdata$species == s)
    if (length(id) == 0) next
    if (nrow(tr) == 0) { out[id] <- mean(train$Y[train$A == a]); next }
    kk <- min(k, nrow(tr))
    D  <- abs(outer(newdata$cp[id], tr$cp, "-"))
    out[id] <- apply(D, 1, function(d) mean(tr$Y[order(d)[seq_len(kk)]]))
  }
  out
}
lin_Q <- function(train, newdata, a) predict(lm(Y ~ A + cp, data = train), transform(newdata, A = a))  # one straight line
logit_g <- function(train, newdata) {                        # each species its own intercept and slope
  fit <- glm(A ~ species + species:cp, family = binomial, data = train)
  bound(predict(fit, newdata, type = "response"))
}

fit_nuisance <- function(train, newdata, Qlearner = "knn", k = 10) {
  if (Qlearner == "knn") { Q1 <- knn_Q(train, newdata, 1, k); Q0 <- knn_Q(train, newdata, 0, k) }
  else { Q1 <- lin_Q(train, newdata, 1); Q0 <- lin_Q(train, newdata, 0) }
  data.frame(Q1 = Q1, Q0 = Q0, g = logit_g(train, newdata))
}
nuisance_full <- function(dat, ...) fit_nuisance(dat, dat, ...)          # train and predict on the same Pokemon
nuisance_crossfit <- function(dat, K = 5, seed = NULL, ...) {             # each prediction comes from the other folds
  if (!is.null(seed)) set.seed(seed)
  fold <- sample(rep(seq_len(K), length.out = nrow(dat)))
  nu <- data.frame(Q1 = NA_real_, Q0 = NA_real_, g = NA_real_)[rep(1, nrow(dat)), ]
  for (k in seq_len(K)) { te <- fold == k; nu[te, ] <- fit_nuisance(dat[!te, ], dat[te, ], ...) }
  rownames(nu) <- NULL
  nu
}

all_estimates <- function(dat, k = 10, K = 5) {
  nf <- nuisance_full(dat, k = k); nc <- nuisance_crossfit(dat, K = K, k = k)
  rbind(naive       = c(est = mean(dat$Y[dat$A == 1]) - mean(dat$Y[dat$A == 0]), se = NA, lo = NA, hi = NA),
        linear      = c(est = unname(coef(lm(Y ~ A + cp, data = dat))["A"]), se = NA, lo = NA, hi = NA),
        plugin_knn  = c(est = est_plugin(nf), se = NA, lo = NA, hi = NA),
        ipw         = c(est = est_ipw(dat, nf), se = NA, lo = NA, hi = NA),
        aipw_full   = est_aipw(dat, nf),
        tmle_full   = est_tmle(dat, nf)[1:4],
        aipw_cf_DML = est_aipw(dat, nc),
        tmle_cf     = est_tmle(dat, nc)[1:4])
}

# ---- only a simulation can do this: split the error into pieces ----
#   plug-in: psi_plug - psi = sampling + first-order bias + empirical process
#   AIPW   : psi_AIPW - psi = (P_n - P)D + R2 + empirical process
# "P" (population mean) is approximated by a large population `pop`
decompose_candy <- function(dat, pop, k = 10, scheme = c("full", "crossfit"), K = 5, seed = 1) {
  scheme <- match.arg(scheme)
  psi <- candy_truth(); n <- nrow(dat)
  set.seed(seed)
  fold <- if (scheme == "full") rep(1L, n) else sample(rep(seq_len(K), length.out = n))
  nu <- data.frame(Q1 = numeric(n), Q0 = numeric(n), g = numeric(n))
  first_order <- 0; R2 <- 0; gerr <- 0; qerr <- 0
  for (f in unique(fold)) {
    te <- fold == f
    tr <- if (scheme == "full") rep(TRUE, n) else !te
    nu[te, ] <- fit_nuisance(dat[tr, ], dat[te, ], k = k)
    np <- fit_nuisance(dat[tr, ], pop, k = k)                 # the same model, evaluated on the population
    w <- mean(te)
    r2_unit <- (np$g - pop$g0) / np$g * (np$Q1 - pop$Q1) + (np$g - pop$g0) / (1 - np$g) * (np$Q0 - pop$Q0)
    fo_unit <- (np$Q1 - pop$Q1) - (np$Q0 - pop$Q0)
    first_order <- first_order + w * mean(fo_unit)
    R2 <- R2 + w * mean(r2_unit)
    if (f == fold[1]) units <- data.frame(species = pop$species, cp = pop$cp, g_err = np$g - pop$g0,
                                          Q1_err = np$Q1 - pop$Q1, Q0_err = np$Q0 - pop$Q0,
                                          R2_unit = r2_unit, first_order_unit = fo_unit)
    gerr <- gerr + w * mean((np$g - pop$g0)^2)
    qerr <- qerr + w * mean(((np$Q1 - pop$Q1)^2 + (np$Q0 - pop$Q0)^2) / 2)
  }
  plug <- mean(nu$Q1 - nu$Q0)
  aipw <- est_aipw(dat, nu)[["est"]]
  samp_plug <- mean(dat$Q1 - dat$Q0) - psi
  samp_aipw <- mean(eif(dat, data.frame(Q1 = dat$Q1, Q0 = dat$Q0, g = dat$g0), psi))   # (P_n - P)D, since PD = 0
  list(plugin = c(total = plug - psi, sampling = samp_plug, first_order = first_order,
                  emp_process = plug - psi - samp_plug - first_order),
       aipw = c(total = aipw - psi, sampling = samp_aipw, R2 = R2, emp_process = aipw - psi - samp_aipw - R2),
       L2 = c(g = sqrt(gerr), Q = sqrt(qerr), product = sqrt(gerr) * sqrt(qerr)),
       units = units)                                         # per-Pokemon errors (first fold's model), for plots
}

# Neyman orthogonality: move g and Q away from the truth by t, compare the two moments
orthogonality_curves <- function(pop, ts = seq(-1, 1, by = 0.05), dq1 = 40, dq0 = -20, dg = 0.6) {
  g <- pop$g0; Q1 <- pop$Q1; Q0 <- pop$Q0
  psi <- mean(Q1 - Q0)
  gbar <- g + dg * g * (1 - g)
  eif_moment <- function(t, move_g, move_Q) {
    gt <- if (move_g) g + t * (gbar - g) else g
    q1 <- if (move_Q) Q1 + t * dq1 else Q1
    q0 <- if (move_Q) Q0 + t * dq0 else Q0
    mean(q1 - q0 + g / gt * (Q1 - q1) - (1 - g) / (1 - gt) * (Q0 - q0)) - psi
  }
  data.frame(t = ts, plugin_Q = ts * (dq1 - dq0),
             eif_Q_only = sapply(ts, eif_moment, move_g = FALSE, move_Q = TRUE),
             eif_g_only = sapply(ts, eif_moment, move_g = TRUE, move_Q = FALSE),
             eif_both   = sapply(ts, eif_moment, move_g = TRUE, move_Q = TRUE))
}

# smallest possible SE: sqrt(V_eff / n), V_eff = E[sigma^2/g + sigma^2/(1-g)] + Var(tau), sigma = 30
candy_se_bound <- function(pop, n) sqrt((mean(30^2 / pop$g0 + 30^2 / (1 - pop$g0)) + var(pop$Q1 - pop$Q0)) / n)

# ---- plotting helpers (base graphics only) ----
COL <- list(red = "#D9480F", teal = "#0B7285", ink = "#1C1B19", muted = "#6B665E",
            gold = "#C98A0C", plum = "#7B3F8C", rule = "#DDD6CA")
lab_par <- function(...) par(mar = c(4.5, 5, 1.5, 1), las = 1, bty = "n", col.axis = COL$muted, ...)

overlap_plot <- function(g_raw, A, xlab = "estimated propensity score  g-hat(X)", labels = c("A = 1", "A = 0"), breaks = 40) {
  br <- seq(0, 1, length.out = breaks + 1)                    # mirrored histogram: treated up, control down
  h1 <- hist(pmin(pmax(g_raw[A == 1], 0), 1), breaks = br, plot = FALSE)$counts
  h0 <- hist(pmin(pmax(g_raw[A == 0], 0), 1), breaks = br, plot = FALSE)$counts
  lab_par()
  plot(NA, xlim = c(0, 1), ylim = c(-max(h0), max(h1)), axes = FALSE, xlab = xlab, ylab = "count")
  axis(1); yt <- pretty(c(-max(h0), max(h1))); axis(2, at = yt, labels = abs(yt))
  rect(br[-length(br)], 0, br[-1], h1, col = adjustcolor(COL$red, .8), border = "white")
  rect(br[-length(br)], 0, br[-1], -h0, col = adjustcolor(COL$teal, .8), border = "white")
  abline(h = 0)
  text(0.98, max(h1) * .85, labels[1], adj = 1, col = COL$red, font = 2)
  text(0.98, -max(h0) * .85, labels[2], adj = 1, col = COL$teal, font = 2)
}

forest_plot <- function(lab, est, lo = NA, hi = NA, ref = NULL, ref_lab = "", col = COL$ink, xlab = "estimate", xlim = NULL) {
  n <- length(est); y <- n:1
  rng <- range(c(est, lo, hi, ref), na.rm = TRUE)
  if (is.null(xlim)) xlim <- rng + c(-1, 1) * diff(rng) * 0.08
  op <- par(mar = c(4.5, min(22, max(8, 0.62 * max(nchar(lab)) + 1)), 1.5, 1), las = 1, bty = "n", col.axis = COL$muted)
  on.exit(par(op))
  plot(NA, xlim = xlim, ylim = c(0.5, n + 0.5), axes = FALSE, xlab = xlab, ylab = "")
  abline(h = y, col = "#F0EBE3")
  if (!is.null(ref)) {
    abline(v = ref, lty = 2, col = COL$muted); pos <- (ref[1] - xlim[1]) / diff(xlim)
    mtext(ref_lab, 3, at = ref[1], adj = if (pos > 0.7) 1 else if (pos < 0.3) 0 else 0.5, cex = .8, col = COL$muted)
  }
  ok <- !is.na(lo)
  segments(lo[ok], y[ok], hi[ok], y[ok], lwd = 3, col = rep_len(col, n)[ok])
  points(est, y, pch = 19, cex = 1.5, col = rep_len(col, n))
  axis(1); axis(2, at = y, labels = lab, tick = FALSE, col.axis = COL$ink)
}

| 格 | 投影片 | 這格回答的問題 |
|---|---|---|
| 01 | 54 | 同一份糖果資料、同一組 nuisance |
| 02–03 | 55–57 | 多一個人，平均動多少？分組平均動多少？（EIF 的兩塊積木） |
| 04 | 58–62 | 拼出 D(O)：它真的平均為 0 嗎？它的變異數是什麼？ |
| 05–06 | 63–65、67–68 | 修法一：加回去（AIPW）；修法二：推 Q̂（TMLE） |
| 07 | 66 | 兩個修法解的是同一個方程式 |

> **以下有一點數學，但每一步都有數字可以對。**看不懂公式也沒關係，只要記得一句話：EIF 告訴你「每一個人把答案往哪邊推、推多大力」；把大家推的力平均起來，就是 plug-in 欠的那一塊。

## 01　同一份資料、同一組 nuisance　<small>（投影片 54）</small>

和 Lab 2 一模一樣：1,500 隻寶可夢，Q̂ 用 kNN（k = 10），ĝ 用 logistic regression。plug-in 是 84.22，真值 80.82。

In [ ]:
truth <- candy_truth()
d  <- simulate_candy(1500, seed = 2026)
nf <- nuisance_full(d, k = 10)
c(truth = truth, plugin = est_plugin(nf))

## 02　積木 1：多一個人，平均動多少？　<small>（投影片 55–56）</small>

最簡單的參數：平均數 μ = E(Y)。現在資料裡多一隻 Y 很高的寶可夢，平均會被往上推。推多少？大約是 (Y<sub>new</sub> − 平均)/n——離平均越遠，推越大力。這個「每個人的推力」`Y − E(Y)`，就是平均數的 influence function。

In [ ]:
y <- d$Y; n <- length(y)
for (y_new in c(300, 800, 1500)) {
  actual <- mean(c(y, y_new)) - mean(y)                       # how much the mean really moves
  approx <- (y_new - mean(y)) / (n + 1)                       # influence function / (n + 1)
  cat(sprintf("new Y = %4d   mean moves %+.3f   (Y_new - mean) / (n + 1) = %+.3f\n", y_new, actual, approx))
}

> **看什麼：**
>
> 三個數字完全對上。Y = 300 的那隻比平均低，把平均往下拉；Y = 1500 的那隻往上推很多。**influence function 其實就是「多一個人，答案動多少」，乘上 n 而已。**

## 03　積木 2：分組平均——只有那一組的人會推動它　<small>（投影片 56–57）</small>

分組平均 E(Y | X = x)，比如「伊布的平均 CP」。多一隻皮皮（Pidgey），伊布的平均不會動；多一隻伊布，才會動。而且伊布越少，每一隻推得越大力：推力是 `I(X = x) / P(X = x) · {Y − E(Y | X = x)}`。

In [ ]:
is_ev <- d$species == "Eevee"; p_ev <- mean(is_ev); m_ev <- mean(d$Y[is_ev])
new_ev <- data.frame(species = "Eevee", Y = 1500); new_pg <- data.frame(species = "Pidgey", Y = 1500)
move <- function(new) { s <- c(as.character(d$species), new$species); yy <- c(d$Y, new$Y); mean(yy[s == "Eevee"]) - m_ev }
rbind(add_an_Eevee  = c(actual = move(new_ev), formula = 1 / p_ev * (1500 - m_ev) / (n + 1)),
      add_a_Pidgey  = c(actual = move(new_pg), formula = 0))
c(share_of_Eevee = p_ev, push_multiplier = 1 / p_ev)

> **看什麼：**
>
> 多一隻伊布，伊布平均動了；多一隻皮皮，完全不動（第一項 I(X = x) = 0）。伊布只占約四分之一，所以每一隻伊布的推力被放大約 4 倍——**那一組越稀有，每個人推得越大力。**等一下你會在 D(O) 裡看到同一個東西：1/g。

## 04　拼出 D(O)：先在四個人身上，再用真的 g、Q 檢查　<small>（投影片 58–62）</small>

用加法規則和乘法規則把兩塊積木拼起來，ATE 的 EIF 是 `D(O; g, Q, ψ) = H(A, X){Y − Q_A(X)} + Q₁(X) − Q₀(X) − ψ`，H = A/g − (1−A)/(1−g)。先用人造的四個人只看 ψ₁ 那一半：預測是 0.4，修正之後是多少？

In [ ]:
toy <- data.frame(A = c(1, 0, 1, 0), Y = c(1, 0, 0, 1), Q1hat = c(.2, .2, .6, .6), ghat = c(.5, .5, .25, .25))
toy$contribution <- with(toy, Q1hat + A * (Y - Q1hat) / ghat)    # one-step contribution for psi_1
toy
c(plugin = mean(toy$Q1hat), one_step = mean(toy$contribution))    # 0.4 -> 0.2
true_nu <- data.frame(Q1 = d$Q1, Q0 = d$Q0, g = d$g0)             # the true nuisances: only a simulation has them
D <- eif(d, true_nu, truth)
pop <- candy_population(20000)
c(mean_D = mean(D), sd_D = sd(D), SE_from_D = sd(D) / sqrt(n), efficiency_bound_SE = candy_se_bound(pop, n))

> **看什麼：**
>
> - 第 3 個人預測 0.6、實際 0，殘差 −0.6，ĝ = 0.25 → 放大 4 倍，把 plug-in 從 0.4 拉到 0.2。個人貢獻可以小於 0，它不是機率。
> - 代真的 g、Q：1,500 隻的 D 平均約 0（只剩抽樣波動）——**平均為 0。**
> - D 的標準差除以 √n 約 1.91，和理論上任何估計器都打不破的下界 `√(V_eff / n)`（約 1.89）幾乎一樣——**變異數就是效率界。**投影片 62 的兩句話，在這裡變成兩個數字。

## 05　修法一：把一階項估出來，加回去（AIPW）　<small>（投影片 63–65、67）</small>

泰勒展開告訴我們：`ψ̂_plug − ψ ≈ −P D(O; ĝ, Q̂, ψ̂_plug)`。真實分布 P 我們拿不到，就用樣本平均 ℙn 代替，把它加回去：`ψ̂ = ψ̂_plug + ℙn D(O; ĝ, Q̂, ψ̂_plug)`。代入 D 的定義，ψ̂_plug 自己消掉，就是 AIPW。

In [ ]:
plug <- est_plugin(nf)
one_step <- plug + mean(eif(d, nf, plug))        # psi_plug + P_n D-hat
H  <- d$A / nf$g - (1 - d$A) / (1 - nf$g); QA <- ifelse(d$A == 1, nf$Q1, nf$Q0)
c(plugin = plug, correction = mean(H * (d$Y - QA)), one_step = one_step, truth = truth)
est_aipw(d, nf)                                   # same number, plus an EIF-based SE and 95% CI
mean(eif(d, nf, one_step))                        # P_n D(O; g-hat, Q-hat, psi_AIPW) = 0

> **看什麼：**
>
> plug-in 84.22 + 修正項 −3.88 = 80.34，真值 80.82——投影片 67 的那條算式。最後一行驗證投影片 63 那句話：AIPW 剛好讓 ℙ<sub>n</sub>D̂ = 0。修正項是什麼？就是第 03 格的推力：每一隻的殘差 Y − Q̂<sub>A</sub>，乘上 1/ĝ（越少見的那一臂，推得越大力），平均起來。

## 06　修法二：不改答案，改 Q̂（TMLE）　<small>（投影片 65、68）</small>

AIPW 是答案算完再加一個修正項。TMLE 反過來：先把 Q̂ 沿著 H 的方向推一點點，推到 ℙ<sub>n</sub>D = 0 為止，再 plug in。`logit Q̂ε(a, X) = logit Q̂(a, X) + ε·H(a, X)`。Y 先縮到 [0, 1]，對 H 做沒有截距、以 logit Q̂ 為 offset 的 logistic regression，估出 ε̂。

In [ ]:
tm <- est_tmle(d, nf); tm
lo <- min(d$Y); hi <- max(d$Y); sc <- function(y) pmin(pmax((y - lo) / (hi - lo), 1e-6), 1 - 1e-6)
QAs <- sc(QA); QAstar <- expit(qlogis(QAs) + tm[["eps"]] * H)        # updated prediction for the observed arm
c(score_before = mean(H * ((d$Y - lo) / (hi - lo) - QAs)), score_after = mean(H * ((d$Y - lo) / (hi - lo) - QAstar)))

> **看什麼：**
>
> ε̂ 很小（約 −0.003），因為 kNN 的 Q̂ 本來就不差。score 從 before 變成 after ≈ 0：logistic fit 的一階條件就是 ℙ<sub>n</sub>[Ĥ(Y − Q̂*<sub>A</sub>)] = 0。答案 80.37，和 AIPW 只差 0.03。

## 07　同一個方程式，兩種解法　<small>（投影片 66）</small>

把 Lab 2、Lab 3 所有的答案放在一起，對著真值看。

In [ ]:
tab <- rbind(naive = c(est = mean(d$Y[d$A == 1]) - mean(d$Y[d$A == 0]), lo = NA, hi = NA),
             linear = c(unname(coef(lm(Y ~ A + cp, data = d))["A"]), NA, NA),
             plugin_kNN = c(plug, NA, NA), IPTW = c(est_ipw(d, nf), NA, NA),
             AIPW = est_aipw(d, nf)[c("est", "lo", "hi")], TMLE = tm[c("est", "lo", "hi")])
round(tab, 2)
keep <- rownames(tab) != "naive"                                  # naive (229) would squash the x axis
forest_plot(rownames(tab)[keep], tab[keep, 1], tab[keep, 2], tab[keep, 3], ref = truth, ref_lab = "truth 80.8",
            col = c(COL$muted, COL$red, COL$gold, COL$teal, COL$teal), xlab = "ATE estimate (CP)")

> **看什麼：**
>
> | | AIPW（one-step） | TMLE（targeting） |
> |---|---|---|
> | 怎麼讓 ℙ<sub>n</sub>D = 0 | 直接把殘差修正的平均加到答案上 | 先推 Q̂，讓殘差修正平均變 0 |
> | 最後的答案 | ψ̂_plug + ℙ<sub>n</sub>D̂ | ℙ<sub>n</sub>(Q̂*₁ − Q̂*₀) |
> | 結果範圍 | 沒有保證（加法修正可能出界） | 尊重 Y 的範圍 |
> | 漸近性質 | 一樣：同一個 D、同一個效率界 | 一樣 |
>
> 圖上：直線、plug-in、IPTW 各自往一邊偏；AIPW、TMLE 的 CI 都蓋到真值。兩個答案幾乎重疊——因為它們解的是同一個方程式。結果範圍那一列，Lab 6（RHC，二元結果）會看到差別。

## 08　帶走一句話

### EIF 就是「每個人把答案推多少」。把推力平均起來加回去是 AIPW；把 Q̂ 推到推力平均為 0 是 TMLE。

你可能會想說：修完就沒有誤差了嗎？沒有。泰勒展開還有一個二階項 R₂，還有「自己改考卷」那一塊。Lab 4 把剩下的東西一塊一塊拆開來看。